# Medical Diagnosis Assistant using RAG + LangChain

**Capstone Project — Healthcare Vertical**

This notebook implements a two-retriever RAG system:

1. **Medical Knowledge Retriever** — answers questions using condition/ICD-10 reference documents.
2. **Patient History Retriever** — embeds historical patient encounters to surface similar past cases for a new patient query.

Both retrievers are orchestrated through LangChain, combined into a single context, and passed to a chat LLM to generate a decision-support style answer with citations.

**Config used in this notebook:**
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector DB: `Chroma`
- Chat model: `gpt-5-nano`

> **Disclaimer:** This is an educational decision-support prototype built on synthetic data. It is **not** a diagnostic tool and must not be used for real clinical decisions.


## 1. Setup & Installation

In [ ]:
# Run this once. Restart the kernel after installing if you hit import errors.
!pip install -q langchain langchain-community langchain-openai langchain-huggingface langchain-chroma chromadb sentence-transformers pandas


In [ ]:
import os
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---- CONFIG ----
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHAT_MODEL_NAME = "gpt-5-nano"

# Set your OpenAI API key (or load from environment / .env file)
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "PASTE_YOUR_KEY_HERE")


## 2. Load & Prepare Data

We merge the encounters file (clinical notes + dx codes) with the patients file (demographics) on `patient_id`.


In [ ]:
encounters = pd.read_csv("healthcare_encounters.csv")
patients = pd.read_csv("healthcare_patients.csv")

df = encounters.merge(patients, on="patient_id", how="left")
print(df.shape)
df.head()


## 3. Build the Medical Knowledge Base

This dataset uses 9 distinct ICD-10 codes. We define short reference write-ups for each — this acts as our
"medical knowledge" corpus for Retriever #1. In a real project you would expand this with content from
sources like MedlinePlus, drug formularies, or clinical guideline summaries.


In [ ]:
ICD10_KNOWLEDGE = {
    "I10": "Essential (primary) hypertension. Chronically elevated blood pressure with no single "
           "identifiable secondary cause. Often linked to diet, weight, stress, and genetics. "
           "Management: lifestyle changes (sodium reduction, exercise, weight loss) and antihypertensive "
           "medications such as ACE inhibitors, ARBs, calcium channel blockers, or thiazide diuretics.",

    "J45": "Asthma. A chronic inflammatory airway disease causing episodic wheezing, shortness of breath, "
           "chest tightness, and cough. Triggers include allergens, exercise, and respiratory infections. "
           "Management: inhaled corticosteroids for control, short-acting beta-agonists (e.g., albuterol) "
           "for acute symptom relief, and trigger avoidance.",

    "M54": "Dorsalgia (back pain), including sciatica. Pain along the back or radiating down the leg due to "
           "nerve root irritation, most commonly from disc issues or muscular strain. Management: physical "
           "therapy, NSAIDs, activity modification, and in chronic/severe cases, imaging and specialist referral.",

    "I25": "Chronic ischemic heart disease. Long-term reduced blood flow to the heart muscle, usually from "
           "atherosclerosis, raising risk of angina and myocardial infarction. Major risk factors include "
           "smoking, hyperlipidemia, hypertension, and diabetes. Management: statins, antiplatelet therapy, "
           "lifestyle changes, and risk factor control.",

    "E78": "Disorders of lipoprotein metabolism (hyperlipidemia). Elevated cholesterol/triglycerides that "
           "increase cardiovascular risk. Often asymptomatic, found on routine labs. Management: statins, "
           "dietary changes, exercise, and monitoring lipid panels.",

    "G47": "Sleep disorders, including insomnia and sleep apnea. Disruption of normal sleep patterns leading "
           "to fatigue and daytime impairment. Management: sleep hygiene counseling, CPAP for sleep apnea, "
           "and addressing contributing conditions like obesity or anxiety.",

    "F41": "Anxiety disorders. Persistent excessive worry or panic symptoms (e.g., palpitations, restlessness) "
           "that can mimic somatic illness. Management: cognitive behavioral therapy, SSRIs/SNRIs, and "
           "lifestyle interventions such as exercise and stress reduction.",

    "K21": "Gastroesophageal reflux disease (GERD). Backflow of stomach acid into the esophagus causing "
           "heartburn, regurgitation, and sometimes atypical chest discomfort. Management: lifestyle changes "
           "(avoid late/large meals, reduce caffeine/alcohol, weight loss, head-of-bed elevation) and "
           "acid-suppressing medications (PPIs, H2 blockers).",

    "E11": "Type 2 diabetes mellitus. Chronic condition with insulin resistance and elevated blood glucose, "
           "associated with obesity and sedentary lifestyle. Risk factor for cardiovascular disease, "
           "neuropathy, and nephropathy. Management: metformin and other glucose-lowering agents, diet, "
           "exercise, and regular A1c monitoring.",
}

knowledge_docs = [
    Document(page_content=text, metadata={"dx_code": code, "source": "knowledge_base"})
    for code, text in ICD10_KNOWLEDGE.items()
]
print(f"Built {len(knowledge_docs)} knowledge documents")


## 4. Build Patient History Documents

Each encounter is turned into a single text "case summary" combining structured demographics with the
free-text clinical note. This is what gets embedded for Retriever #2 (similar historical case lookup).


In [ ]:
def build_case_text(row):
    smoker = "smoker" if row["smoker"] == 1 else "non-smoker"
    diabetic = "diabetic" if row["diabetic"] == 1 else "non-diabetic"
    return (
        f"Patient: {row['age']}yo {row['sex']}, {smoker}, {diabetic}, BMI {row['bmi']}. "
        f"Note: {row['note']} Diagnosis code: {row['dx_code']}."
    )

df["case_text"] = df.apply(build_case_text, axis=1)

history_docs = [
    Document(
        page_content=row["case_text"],
        metadata={
            "encounter_id": row["encounter_id"],
            "patient_id": row["patient_id"],
            "dx_code": row["dx_code"],
            "date": row["date"],
            "source": "patient_history",
        },
    )
    for _, row in df.iterrows()
]
print(f"Built {len(history_docs)} patient history documents")
history_docs[0].page_content


## 5. Create Embeddings & Vector Stores

We use a single embedding model (`all-MiniLM-L6-v2`) for both corpora but store them in **two separate
Chroma collections** so we can query knowledge and patient history independently or together.


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

knowledge_vectorstore = Chroma.from_documents(
    documents=knowledge_docs,
    embedding=embedding_model,
    collection_name="medical_knowledge",
    persist_directory="./chroma_db",
)

history_vectorstore = Chroma.from_documents(
    documents=history_docs,
    embedding=embedding_model,
    collection_name="patient_history",
    persist_directory="./chroma_db",
)

knowledge_retriever = knowledge_vectorstore.as_retriever(search_kwargs={"k": 2})
history_retriever = history_vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector stores ready.")


## 6. Quick Retrieval Sanity Checks

Before wiring up the LLM, confirm both retrievers return sensible results on their own.


In [ ]:
print("--- Knowledge retriever test ---")
for d in knowledge_retriever.invoke("What lifestyle changes help with GERD?"):
    print(d.metadata["dx_code"], "->", d.page_content[:100], "...")

print("\n--- Patient history retriever test ---")
for d in history_retriever.invoke(
    "65 year old male smoker with fatigue and history of hyperlipidemia"
):
    print(d.metadata["encounter_id"], d.metadata["dx_code"], "->", d.page_content[:120], "...")


## 7. Build the LangChain RAG Chain

The chain:
1. Takes a user query (patient presentation / question).
2. Retrieves top documents from **both** vector stores.
3. Merges them into a single context block, clearly labeled by source type.
4. Sends the context + query to `gpt-5-nano` with a prompt that enforces citation and a decision-support disclaimer.


In [ ]:
llm = ChatOpenAI(model=CHAT_MODEL_NAME, temperature=0)

SYSTEM_PROMPT = """You are a clinical decision-support assistant. You are NOT a doctor and must never
state a definitive diagnosis. Use only the provided context (medical knowledge excerpts and similar
historical patient cases) to answer.

Rules:
- Cite which source(s) you used, referencing dx_code and/or encounter_id from the context.
- If the retrieved context is weak or irrelevant to the question, say so explicitly and recommend
  clinical evaluation instead of guessing.
- Always end your answer with: "This is a decision-support suggestion, not a diagnosis."
"""

PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",
     "Medical Knowledge Context:\n{knowledge_context}\n\n"
     "Similar Patient History Context:\n{history_context}\n\n"
     "Question:\n{question}"),
])

def format_docs(docs, empty_msg="No relevant documents found."):
    if not docs:
        return empty_msg
    return "\n\n".join(
        f"[{d.metadata.get('dx_code', d.metadata.get('source'))} | "
        f"{d.metadata.get('encounter_id', 'knowledge_base')}] {d.page_content}"
        for d in docs
    )

def retrieve_and_format(question):
    knowledge_docs_retrieved = knowledge_retriever.invoke(question)
    history_docs_retrieved = history_retriever.invoke(question)
    return {
        "knowledge_context": format_docs(knowledge_docs_retrieved),
        "history_context": format_docs(history_docs_retrieved),
        "question": question,
    }

rag_chain = (
    RunnablePassthrough()
    | (lambda q: retrieve_and_format(q))
    | PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")


## 8. Demo Queries

Try the chain on a few example questions — pure knowledge lookup, pure case-similarity lookup, and a
combined query.


In [ ]:
demo_questions = [
    "What is essential hypertension and how is it typically managed?",
    "65-year-old male, smoker, reports fatigue for the past week, history of hyperlipidemia. What should be considered?",
    "Patient denies chest pain but has episodic back pain and is on medication for hyperlipidemia - could this be cardiac related?",
]

for q in demo_questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print("-" * 80)


## 9. Evaluation: Top-K Retrieval Accuracy

A simple, quantifiable way to evaluate the patient-history retriever: hold out a sample of encounters,
treat their notes as "new patient" queries, retrieve the most similar *other* cases, and check whether
the majority dx_code among neighbors matches the true dx_code of the held-out case.


In [ ]:
import random
from collections import Counter

random.seed(42)
test_sample = df.sample(n=50, random_state=42)

correct = 0
for _, row in test_sample.iterrows():
    query_text = build_case_text(row)
    retrieved = history_vectorstore.similarity_search(query_text, k=4)
    # Exclude the exact same encounter if it comes back (it will, since it's in the index)
    retrieved = [d for d in retrieved if d.metadata["encounter_id"] != row["encounter_id"]][:3]
    if not retrieved:
        continue
    predicted_codes = [d.metadata["dx_code"] for d in retrieved]
    majority_code = Counter(predicted_codes).most_common(1)[0][0]
    if majority_code == row["dx_code"]:
        correct += 1

accuracy = correct / len(test_sample)
print(f"Top-3 majority-vote retrieval accuracy: {accuracy:.2%} (on {len(test_sample)} held-out cases)")


## 10. Edge Case Handling

Demonstrate the system abstaining gracefully when there's no good match — important for a
responsible-AI discussion in your report.


In [ ]:
edge_case_question = "Patient presents with sudden vision loss and severe headache."
print("Q:", edge_case_question)
print("A:", rag_chain.invoke(edge_case_question))


## 11. Notes / Next Steps

- **Limitations:** Built on synthetic data with only 9 dx codes; not representative of real clinical diversity.
- **Responsible AI:** This tool is for decision support and education only — not a diagnostic device.
  Real deployment would require clinical validation, bias auditing (e.g., check if `smoker`/`diabetic`
  flags skew retrieval), and regulatory review (HIPAA, FDA SaMD classification as applicable).
- **Possible extensions:**
  - Swap the hand-written knowledge base for a larger external corpus (e.g., MedlinePlus articles).
  - Add a re-ranker on top of the initial vector retrieval.
  - Build a Streamlit front-end for clinician-style interaction.
  - Add conversational memory (LangChain `RunnableWithMessageHistory`) for multi-turn refinement.
